In [ ]:
import cv2
import os
import glob
import numpy as np
from wisardpkg import ClusWisard
import sys
sys.path.append('/mnt/c/Users/Isabella/tcc')
from wisard_object_tracker.src.utils import tracker_utils
import matplotlib.pyplot as plt

DATASET_ROOT_FOLDER = '/mnt/c/Users/Isabella/TCC/wisard_object_tracker/data/tiger2'
IMAGE_FOLDER = os.path.join(DATASET_ROOT_FOLDER, 'imgs')
GT_TXT_PATH = os.path.join(DATASET_ROOT_FOLDER, 'tiger2_gt.txt')

In [ ]:
# Carrega imagens
image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, '*.png')))
print(f"Encontradas {len(image_paths)} imagens")

# Carrega ground truths
all_ground_truths = tracker_utils.load_ground_truth_from_gt_txt(GT_TXT_PATH)

# Primeiro frame

In [ ]:
# Carrega primeiro frame  # lembrar de carregar certo em tons de cinza
first_frame = cv2.imread(image_paths[0])
print(f"Primeiro frame carregado: {first_frame.shape}")

first_gt = all_ground_truths[0]
print(f"Ground Truth do primeiro frame: {first_gt}")

# Mostra o frame com a bbox
first_frame_with_bbox = first_frame.copy()
x, y, w, h = map(int, first_gt)
cv2.rectangle(first_frame_with_bbox, (x, y), (x + w, y + h), (0, 255, 0), 2) 

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(first_frame_with_bbox, cv2.COLOR_BGR2RGB))
plt.title('Primeiro Frame - Objeto a Rastrear')
plt.axis('off')
plt.show()


In [ ]:
# --- PASSO 2: EXTRAIR E PRÉ-PROCESSAR PATCH ---
print("\n🔍 PASSO 2: Extraindo e pré-processando patch...")

# Extrai patch do objeto
patch = tracker_utils.extract_patch(first_frame, first_gt)

# Mostra o patch ORIGINAL (antes do pré-processamento)
plt.figure(figsize=(8, 6))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(patch, cv2.COLOR_BGR2RGB))
plt.title(f'Patch Original\n shape:{patch.shape}')
plt.axis('off')


In [ ]:
# Pré-processa o patch
import cv2
import numpy as np

def preprocess_frame(frame):
    # Converte para grayscale caso seja RGB
    if len(frame.shape) == 3:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        gray = frame.copy()
    # piorou apenas com otsu sem média global
    # 1) MÉDIA GLOBAL  (correção do erro)
    global_mean = int(np.mean(gray))     # precisa ser escalar Python
    no_bg_global = cv2.subtract(gray, global_mean)

    # 2) Binarização Otsu
    _, otsu = cv2.threshold(
        no_bg_global, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # 3) Retornar vetor flatten binário
    vector = (otsu.flatten() > 0).astype(np.uint8)

    return vector
    

In [ ]:
# Pré-processa o patch
first_pattern = preprocess_frame(patch)
print(f"Primeiro Pattern pré-processado: {first_pattern.shape}")

In [ ]:
# --- PASSO 3: INSTANCIANDO CLUSWISARD ---

# Parâmetros simples
parameters = {
    "INPUT_PATTERN_SIDE": 32,                  
    "CLUSWISARD_ADDRESS_SIZE": 7,          
    "ACCEPTABLE_ACTIVATION_SCORE": 0.6,       
    "MIN_ACCEPTABLE_ACTIVATION_SCORE": 0.1,    
    "CLUSWISARD_MIN_SCORE": 0.4,               
    "CLUSWISARD_THRESHOLD": 3,                 
    "CLUSWISARD_DISCRIMINATOR_LIMIT": 3,      
    "CLUSWISARD_BLEACHING_ACTIVATED": False,     
    "CLUSWISARD_ACTIVATION_DEGREE": True,
    "CLUSWISARD_RETURN_CONFIDENCE": True,
    "CLUSWISARD_CLASSES_DEGREES": True,
    "INITIAL_FRAME": 1,
    "LAST_FRAME": 25,
    "MAX_SEARCH_WINDOW_SCALE": 2,
}

# Instancia ClusWisard

clus = ClusWisard(
            parameters['CLUSWISARD_ADDRESS_SIZE'], # address size
            parameters['CLUSWISARD_MIN_SCORE'], # min score
            parameters['CLUSWISARD_THRESHOLD'], # threshold
            parameters['CLUSWISARD_DISCRIMINATOR_LIMIT'], # discriminator limit
            bleachingActivated=parameters['CLUSWISARD_BLEACHING_ACTIVATED'],
            returnActivationDegree=parameters['CLUSWISARD_ACTIVATION_DEGREE'],
            returnConfidence=parameters['CLUSWISARD_RETURN_CONFIDENCE'],
            returnClassesDegrees=parameters['CLUSWISARD_CLASSES_DEGREES']
        )

In [ ]:
# Treina com o patch do objeto
clus.train([first_pattern.tolist()], ["object"])
print("✅ ClusWisard treinado com o patch do primeiro frame!")

# Região de Busca

In [ ]:
import numpy as np  # importa o NumPy para operações numéricas e trigonométricas

def generate_search_regions_circular(
    prev_bbox, 
    frame_shape, 
    search_region_scale=1.5, 
    step_size=1,
    start_angle=0,
    clockwise=True
):
    """
    Gera regiões de busca circulares em torno do bbox anterior,
    onde step_size define o deslocamento em pixels reais
    (1 px = step_size=1).
    """

    x, y, w, h = prev_bbox
    center_x, center_y = x + w // 2, y + h // 2

    raio_max = (max(w, h) / 2) * search_region_scale

    yield (x, y, w, h)  # primeiro retorna o bbox original

    # passo radial em pixels
    for raio in np.arange(step_size, raio_max + step_size, step_size):
        circunferencia = 2 * np.pi * raio

        # define espaçamento angular de forma que os pontos na borda
        # fiquem separados por ~step_size pixels ao longo da circunferência
        num_steps = max(8, int(np.ceil(circunferencia / step_size)))
        direction = -1 if clockwise else 1

        thetas = start_angle + direction * np.linspace(0, 2 * np.pi, num_steps, endpoint=False)
        
        # deslocamentos em pixels
        pxs = (center_x + raio * np.cos(thetas) - w / 2).astype(int)
        pys = (center_y + raio * np.sin(thetas) - h / 2).astype(int)

        # garante que o bbox não ultrapasse os limites do frame
        pxs = np.clip(pxs, 0, frame_shape[1] - w)
        pys = np.clip(pys, 0, frame_shape[0] - h)

        for px, py in zip(pxs, pys):
            yield (px, py, w, h)


In [ ]:
def show_top_regions_grid(frame_path, results_sorted, top_n=500, cols=10):
    """
    Mostra as N regiões mais bem classificadas em um grid.

    Parâmetros:
        frame: imagem original (BGR)
        results_sorted: lista retornada de classify_search_regions()
        top_n: número de regiões a exibir
        cols: número de colunas no grid
    """
    frame = cv2.imread(frame_path)
    top_n = min(top_n, len(results_sorted))
    rows = (top_n + cols - 1) // cols

    plt.figure(figsize=(4 * cols, 4 * rows))

    h_frame, w_frame = frame.shape[:2]

    for i in range(top_n):
        region_info = results_sorted[i]
        region_id = region_info["region_id"]
        activation = region_info["activation"]

        # --- ✅ Converte coordenadas para inteiros
        x, y, w, h = [int(round(v)) for v in region_info["bbox"]]

        # --- ⚠️ Garante que os índices estão dentro dos limites do frame
        x = max(0, min(x, w_frame - 1))
        y = max(0, min(y, h_frame - 1))
        w = max(1, min(w, w_frame - x))
        h = max(1, min(h, h_frame - y))

        # --- Extrai o patch da região
        patch = frame[y:y+h, x:x+w]
        if patch.size == 0:
            print(f"⚠️ Region {region_id} vazia - ignorando exibição")
            continue

        patch_rgb = cv2.cvtColor(patch, cv2.COLOR_BGR2RGB)

        # --- Mostra patch no grid
        plt.subplot(rows, cols, i + 1)
        plt.imshow(patch_rgb)
        plt.title(f"#{i+1} Reg {region_id}\nAct={activation:.3f}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()


# Analise de parâmetros da janela de busca

In [ ]:
def _show_top_regions_grid(results, N=9):
    """
    Exibe um grid com as N melhores regiões ordenadas por ativação.
    """

    # Ordenar em ordem decrescente
    sorted_results = sorted(results, key=lambda r: r['activation'], reverse=True)

    top = sorted_results[:N]
    n = len(top)

    cols = 3
    rows = math.ceil(n / cols)

    plt.figure(figsize=(12, 4 * rows))
    plt.suptitle(f"{n} melhores regiões (ordenadas por ativação)", fontsize=16)

    for idx, r in enumerate(top):
        plt.subplot(rows, cols, idx + 1)
        plt.imshow(r["patch"])
        plt.title(f"{r['activation']:.4f}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

def _show_frame_with_regions(frame_rgb, results, best_bbox):
    """
    Desenha no frame:
    - TODAS as regiões testadas em vermelho
    - A melhor região em verde
    """

    frame_copy = frame_rgb.copy()

    # desenhar todas as regiões testadas em vermelho
    for r in results:
        x, y, w, h = r["bbox"]

        x, y, w, h = int(x), int(y), int(w), int(h)

        cv2.rectangle(frame_copy, (x, y), (x + w, y + h), (255, 0, 0), 1)

    # destaque da melhor região (verde)
    bx, by, bw, bh = best_bbox
    bx, by, bw, bh = int(bx), int(by), int(bw), int(bh)

    cv2.rectangle(frame_copy, (bx, by), (bx + bw, by + bh), (0, 255, 0), 2)

    plt.figure(figsize=(10, 6))
    plt.title("Frame com todas as regiões testadas")
    plt.imshow(frame_copy)
    plt.axis("off")
    plt.show()


import cv2
import matplotlib.pyplot as plt
import math

def tracker(frame_path, frame_id, prev_bbox, N=9):
    """
    Realiza tracking em um único frame.

    - frame_path: caminho da imagem
    - frame_id: índice do frame
    - prev_bbox: bounding-box anterior (x, y, w, h)
    - clus: modelo ClusWisard já carregado/treinado
    - N: número de melhores regiões para exibir no grid

    Retorna:
        best_bbox (tuple)
        best_activation (float)
        num_tested_regions (int)
    """

    print(f"\n📷 Frame {frame_id}: {frame_path}")

    # === Ler frame ===
    frame = cv2.imread(frame_path)
    if frame is None:
        print(f"❌ Frame não carregado!")
        return prev_bbox, -1, 0

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # === Gerar regiões de busca ===
    print("🔍 Gerando regiões de busca...")
    search_gen = generate_search_regions_circular(
        prev_bbox=prev_bbox,
        frame_shape=frame.shape,
        search_region_scale=2,
        step_size=1
    )

    results = []
    num_tested = 0

    # === LOOP: testar regiões até achar alguma com ativação > 0.7 ===
    for region in search_gen:
        num_tested += 1

        patch_region = tracker_utils.extract_patch(frame, region)
        if patch_region.size == 0:
            continue

        pattern = preprocess_frame(patch_region)
        cls_result = clus.classify([pattern.tolist()])[0]
        activation = cls_result.get("activationDegree", -1)

        results.append({
            "bbox": region,
            "activation": activation,
            "patch": cv2.cvtColor(patch_region, cv2.COLOR_BGR2RGB),
            "pattern": pattern
        })

        # critério de parada
        if activation > 0.6:
            print(f"🎯 Ativação > 0.7 encontrada após {num_tested} regiões!")
            break

    if len(results) == 0:
        print("⚠️ Nenhuma região válida encontrada. Mantendo prev_bbox.")
        return prev_bbox, -1, 0

    # === Selecionar melhor região ===
    best_region = max(results, key=lambda r: r["activation"])
    best_bbox = best_region["bbox"]
    best_activation = best_region["activation"]
    best_pattern = best_region["pattern"]

    if best_activation < 0.4:
        print(f"🎯 Retreino!")
        clus.train([best_pattern.tolist()], ["object"])


    print(f"🏆 Melhor região: {best_bbox}, ativ={best_activation:.4f}")
    print(f"🔢 Regiões testadas: {num_tested}")
    print(f"🔢 Resultado: {cls_result}")

    # === Mostrar frame com TODAS as regiões testadas ===
    _show_frame_with_regions(rgb_frame, results, best_bbox)

    # === Grid das N melhores regiões ===
    _show_top_regions_grid(results, N)

    return best_bbox, best_activation, num_tested


Segundo Frame

In [ ]:
frame_path=image_paths[1]
frame_idx=2
prev_bbox = (16.0, 30.0, 34.0, 39.0)

prev_bbox2, results2, num_tested = tracker(frame_path, frame_idx, prev_bbox)


In [ ]:
frame_path=image_paths[2]
frame_idx=3
prev_bbox3, results3, num_tested = tracker(frame_path, frame_idx, prev_bbox2)

In [ ]:
frame_path=image_paths[3]
frame_idx=4
prev_bbox4, results4, num = tracker(frame_path, frame_idx, prev_bbox3)

In [ ]:
frame_path=image_paths[4]
frame_idx=5
prev_bbox5, results5, num = tracker(frame_path, frame_idx, prev_bbox4)

In [ ]:
frame_path=image_paths[5]
frame_idx=6
prev_bbox6, results6, num = tracker(frame_path, frame_idx, prev_bbox5)

In [ ]:
frame_path=image_paths[6]
frame_idx=7
prev_bbox7, results7, num = tracker(frame_path, frame_idx, prev_bbox6)

In [ ]:
frame_path=image_paths[7]
frame_idx=8
prev_bbox8, results8, num = tracker(frame_path, frame_idx, prev_bbox7)

In [ ]:
frame_path=image_paths[8]
frame_idx=9
prev_bbox9, results9, num = tracker(frame_path, frame_idx, prev_bbox8)

In [ ]:
frame_path=image_paths[9]
frame_idx=10
prev_bbox10, results10, num = tracker(frame_path, frame_idx, prev_bbox9)

In [ ]:
frame_path=image_paths[10]
frame_idx=11
prev_bbox11, results11, num = tracker(frame_path, frame_idx, prev_bbox10)

In [ ]:
frame_path=image_paths[11]
frame_idx=12
prev_bbox12,  results11, num = tracker(frame_path, frame_idx, prev_bbox11)

In [ ]:
frame_path=image_paths[12]
frame_idx=13
prev_bbox13, results13, num = tracker(frame_path, frame_idx, prev_bbox12)

In [ ]:
frame_path=image_paths[13]
frame_idx=14
prev_bbox14, results14, num  = tracker(frame_path, frame_idx, prev_bbox13)

In [ ]:
frame_path=image_paths[14]
frame_idx=15
prev_bbox15, results15, num   = tracker(frame_path, frame_idx, prev_bbox14)

In [ ]:
frame_path=image_paths[15]
frame_idx=16
prev_bbox16, results16, num   = tracker(frame_path, frame_idx, prev_bbox15)

In [ ]:
frame_path=image_paths[16]
frame_idx=17
prev_bbox17, results17, num   = tracker(frame_path, frame_idx, prev_bbox16)

In [ ]:
frame_path=image_paths[17]
frame_idx=18
prev_bbox18, results18, num   = tracker(frame_path, frame_idx, prev_bbox17)

In [ ]:
frame_path=image_paths[18]
frame_idx=19
prev_bbox19, results19, num   = tracker(frame_path, frame_idx, prev_bbox18)

In [ ]:
frame_path=image_paths[19]
frame_idx=20
prev_bbox20, results20, num   = tracker(frame_path, frame_idx, prev_bbox19)

In [ ]:
frame_path=image_paths[19]
frame_idx=20
prev_bbox20, results20, num   = tracker(frame_path, frame_idx, prev_bbox19)

In [ ]:
frame_path=image_paths[19]
frame_idx=20
prev_bbox20, results20, num   = tracker(frame_path, frame_idx, prev_bbox19)

In [ ]:
frame_path=image_paths[20]
frame_idx=21
prev_bbox21, results21, num   = tracker(frame_path, frame_idx, prev_bbox20)

In [ ]:
frame_path=image_paths[21]
frame_idx=22
prev_bbox22, results22, num   = tracker(frame_path, frame_idx, prev_bbox21)

In [ ]:
frame_path=image_paths[22]
frame_idx=23
prev_bbox23, results23, num   = tracker(frame_path, frame_idx, prev_bbox22)

In [ ]:
frame_path=image_paths[23]
frame_idx=24
prev_bbox24, results24, num   = tracker(frame_path, frame_idx, prev_bbox23)

In [ ]:
frame_path=image_paths[24]
frame_idx=25
prev_bbox25, results25, num   = tracker(frame_path, frame_idx, prev_bbox24)

In [ ]:
frame_path=image_paths[25]
frame_idx=26
prev_bbox26, results26, num   = tracker(frame_path, frame_idx, prev_bbox25)

In [ ]:
frame_path=image_paths[26]
frame_idx=27
prev_bbox27, results27, num   = tracker(frame_path, frame_idx, prev_bbox26)

In [ ]:
frame_path=image_paths[27]
frame_idx=28
prev_bbox28, results28, num   = tracker(frame_path, frame_idx, prev_bbox27)

In [ ]:
frame_path=image_paths[28]
frame_idx=29
prev_bbox29, results29, num   = tracker(frame_path, frame_idx, prev_bbox28)

In [ ]:
frame_path=image_paths[29]
frame_idx=30
prev_bbox30, results30, num   = tracker(frame_path, frame_idx, prev_bbox29)

In [ ]:
frame_path=image_paths[30]
frame_idx=31
prev_bbox31, results31, num   = tracker(frame_path, frame_idx, prev_bbox30)

In [ ]:
frame_path=image_paths[31]
frame_idx=32
prev_bbox32, results32, num   = tracker(frame_path, frame_idx, prev_bbox31)

In [ ]:
frame_path=image_paths[32]
frame_idx=33
prev_bbox33, results33, num   = tracker(frame_path, frame_idx, prev_bbox32)

In [ ]:
frame_path=image_paths[33]
frame_idx=34
prev_bbox34, results34, num   = tracker(frame_path, frame_idx, prev_bbox33)

In [ ]:
frame_path=image_paths[34]
frame_idx=35
prev_bbox35, results35, num   = tracker(frame_path, frame_idx, prev_bbox34)

In [ ]:
frame_path=image_paths[35]
frame_idx=36
prev_bbox36, results36, num   = tracker(frame_path, frame_idx, prev_bbox35)

In [ ]:
frame_path=image_paths[36]
frame_idx=37
prev_bbox37, results37, num   = tracker(frame_path, frame_idx, prev_bbox36)